# Model Governance Implementation

## What You Will Learn
- Understand model governance framework concepts
- Implement access control patterns (RBAC) for model registries
- Apply model versioning policies and approval workflows
- Use MLflow API to manage model stages (Staging → Production)

## Why This Matters in MLOps
Without governance, anyone can deploy any model to production — a recipe for disaster. Governance ensures that models are reviewed, approved, and tracked. This is essential for regulated industries (finance, healthcare) and for maintaining trust in ML systems.

## You're Done When...
- You can explain RBAC in the context of ML model registries
- You have searched MLflow experiments by tags
- You have transitioned a model version from Staging to Production
- You have completed the fill-in-the-blank exercises

---

### MLOps Perspective

In enterprise MLOps, the model registry is the source of truth. Governance means:
- **Who** can register models (data scientists)
- **Who** can promote models (ML engineers / reviewers)
- **Who** can deploy to production (DevOps / SRE)
- **What** evidence is required (performance metrics, fairness checks)

MLflow's Model Registry supports stage transitions, versioning, and tags that enable these governance patterns.

---
## 1. Model Governance Framework Concepts

A governance framework defines:
- **Policies**: Rules for model development, testing, and deployment
- **Roles**: Data Scientist, ML Engineer, Reviewer, Auditor
- **Gates**: Checkpoints that a model must pass (e.g., accuracy threshold, fairness audit)
- **Audit Trail**: Immutable record of who did what and when

### MLflow Model Stages
| Stage | Description |
|-------|-------------|
| None | Unregistered or newly registered |
| Staging | Ready for review/testing |
| Production | Approved for live serving |
| Archived | Retired from active use |

---
## 2. Access Control Patterns (RBAC)

Role-Based Access Control restricts registry actions based on user roles.

| Role | Register | Transition to Staging | Transition to Production | Archive |
|------|----------|----------------------|------------------------|--------|
| Data Scientist | ✅ | ✅ | ❌ | ❌ |
| Reviewer | ✅ | ✅ | ✅ | ❌ |
| Admin | ✅ | ✅ | ✅ | ✅ |

In MLflow, RBAC is enforced by the tracking server. We simulate it here using tags.

In [ ]:
import mlflow
from mlflow import MlflowClient
import time

# Configure MLflow tracking URI (use local or remote)
mlflow.set_tracking_uri('http://localhost:5000')
client = MlflowClient()

# Simulate RBAC by tagging experiments with allowed roles
experiment_name = 'Governance_Demo_' + str(int(time.time()))
try:
    experiment_id = mlflow.create_experiment(
        experiment_name,
        tags={'allowed_roles': 'data_scientist,reviewer,admin'}
    )
    print(f'Created experiment: {experiment_name} (ID: {experiment_id})')
except Exception as e:
    print(f'Experiment may already exist: {e}')

## 3. Search Experiments by Tags

Tag-based search enables policy enforcement: only show experiments the user is allowed to access.

In [ ]:
# Search for experiments with specific tags
user_role = 'data_scientist'

experiments = client.search_experiments(
    filter_string=f"tags.allowed_roles LIKE '%{user_role}%'"
)

print(f'Experiments accessible by {user_role}:')
for exp in experiments:
    print(f'  - {exp.name} (ID: {exp.experiment_id})')

In [ ]:
# Log a model to the registry for governance demo
with mlflow.start_run(experiment_id=experiment_id) as run:
    mlflow.log_metric('accuracy', 0.94)
    mlflow.log_param('model_type', 'RandomForest')
    mlflow.sklearn.log_model(
        sk_model=None,  # placeholder
        artifact_path='model',
        registered_model_name='ForecastModel_Governance'
    )
    run_id = run.info.run_id
    print(f'Run ID: {run_id}')

---
## 4. Model Versioning and Approval Workflows

Stage transitions model the approval workflow: None → Staging → Production → Archived.

In [ ]:
# Get the registered model and its versions
model_name = 'ForecastModel_Governance'
try:
    versions = client.get_latest_versions(model_name, stages=['None'])
    if versions:
        version = versions[0]
        print(f'Found model version: {version.version} in stage: {version.current_stage}')
        
        # Transition to Staging
        client.transition_model_version_stage(
            name=model_name,
            version=version.version,
            stage='Staging'
        )
        print(f'Model version {version.version} transitioned to Staging')
        
        # Add governance description
        client.set_model_version_tag(
            name=model_name,
            version=version.version,
            key='review_status',
            value='pending'
        )
        print('Review status tag set to "pending"')
except Exception as e:
    print(f'Note: {e}')
    print('(This is expected if no model was registered. Move to next cell.)')

In [ ]:
# Simulate approval: promote from Staging to Production
model_name = 'ForecastModel_Governance'
try:
    staging_versions = client.get_latest_versions(model_name, stages=['Staging'])
    if staging_versions:
        version = staging_versions[0]
        
        # Reviewer approves
        client.set_model_version_tag(
            name=model_name,
            version=version.version,
            key='review_status',
            value='approved'
        )
        client.set_model_version_tag(
            name=model_name,
            version=version.version,
            key='reviewed_by',
            value='reviewer@example.com'
        )
        
        # Promote to Production
        client.transition_model_version_stage(
            name=model_name,
            version=version.version,
            stage='Production'
        )
        print(f'Model version {version.version} approved and promoted to Production!')
        
        # Verify
        prod_versions = client.get_latest_versions(model_name, stages=['Production'])
        print(f'Production versions: {[v.version for v in prod_versions]}')
except Exception as e:
    print(f'Note: {e}')

---
## 5. Fill-in-the-Blank Exercises

Complete the code below by replacing `___` with the correct MLflow API calls.

In [ ]:
# Exercise 1: Search experiments by tag
from mlflow import MlflowClient

client = MlflowClient()
user_dept = 'engineering'

# Search for experiments tagged with department
experiments = client.____(  # Hint: use search_experiments
    filter_string=f"tags.department LIKE '%{user_dept}%'"
)

print(f'Found {len(experiments)} experiments for {user_dept}')
for exp in experiments:
    print(f'  - {exp.name}')

In [ ]:
# Exercise 2: Transition model version stage
model_name = 'ForecastModel_Governance'

# Get latest staging version
staging_versions = client.get_latest_versions(model_name, stages=['Staging'])
if staging_versions:
    v = staging_versions[0]
    
    # Transition to Production
    client.____(  # Hint: use transition_model_version_stage
        name=model_name,
        version=v.version,
        stage='Production'
    )
    print(f'Version {v.version} promoted to Production')
else:
    print('No staging versions found')

---
## Summary

- Model governance defines policies, roles, and gates for the ML lifecycle
- RBAC restricts model registry actions based on user roles
- MLflow tags enable search and policy enforcement
- Stage transitions (Staging → Production) model the approval workflow

In the next notebook, you will implement audit logging to track all ML system activities.